In [ ]:
# SPLIT DATASET INTO STAR SCHEMA
# FACT TABLE FIELDS: Order ID, item quantities, 

In [9]:
import os
import hashlib
import pandas as pd

# 1. Load the raw dataset
# Replace with your actual path to the DataCo dataset
raw_data_path = "../data/cleaned/cleaned_data.csv"
df = pd.read_csv(raw_data_path, encoding="utf-8")

# Create output directory for processed files
output_dir = "../data/processed"

In [ ]:
# CREATE DIM_CUSTOMERS

dim_customers = (
    df[["Customer Id", "Customer Fname", "Customer Lname", "Customer Segment"]]
    .drop_duplicates(subset=["Customer Id"])
    .reset_index(drop=True)
)
dim_customers.to_csv(f"{output_dir}/dim_customers.csv", index=False)

In [22]:
# CREATE DIM_PRODUCTS

dim_products = (
    df[
        [
            "Product Card Id",
            "Product Name",
            "Category Name",
            "Department Name",
            "Product Price",
        ]
    ]
    .drop_duplicates(subset=["Product Card Id"])
    .reset_index(drop=True)
)
dim_products.to_csv(f"{output_dir}/dim_products.csv", index=False)
dim_products

,Product Card Id,Product Name,Category Name,Department Name,Product Price
0,1360,Smart watch,Sporting Goods,Fitness,327.750000
1,365,Perfect Fitness Perfect Rip Deck,Cleats,Apparel,59.990002
2,627,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,Golf,39.990002
3,502,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,Golf,50.000000
4,278,Under Armour Men's Compression EV SL Slide,Electronics,Footwear,44.990002
...,...,...,...,...,...
113,646,Merrell Women's Grassbow Sport Hiking Shoe,Men's Golf Clubs,Outdoors,99.989998
114,1361,Toys,Toys,Fan Shop,11.540000
115,1073,Pelican Sunstream 100 Kayak,Water Sports,Fan Shop,199.990005
116,1059,Pelican Maverick 100X Kayak,Water Sports,Fan Shop,349.989990


In [16]:
# CREATE DIM_LOCATION

# Generate deterministic MD5 hash string using location identifying columns
def generate_location_key(row):
    input_str = (
        f"{row['Order Region']}|{row['Order Country']}|{row['Order State']}|{row['Order City']}"
    )
    # generate MD5 hex digest: 32 char long 
    return hashlib.md5(input_str.encode('utf-8')).hexdigest()
    
# Grouping unique combinations of delivery locations
dim_location = (
    df[["Order Region", "Order Country", "Order State", "Order City"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Generate unique location key using hash function
dim_location["Location_Key"] = dim_location.apply(generate_location_key, axis=1)

# Reorder columns to put Key first
dim_location = dim_location[
    ["Location_Key", "Order Region", "Order Country", "Order State", "Order City"]
]
dim_location.to_csv(f"{output_dir}/dim_location.csv", index=False)

In [21]:
# CREATE DIM_SHIPPING
def generate_shipping_key(row):
    input_str = (
        f"{row['Shipping Mode']}|{row['Delivery Status']}"
    )
    # generate MD5 hex digest: 32 char long 
    return hashlib.md5(input_str.encode('utf-8')).hexdigest()
        
dim_shipping = df[["Shipping Mode", "Delivery Status"]].drop_duplicates().reset_index(drop=True)
dim_shipping["Shipping_Mode_Key"] = dim_shipping.apply(generate_shipping_key, axis=1)
dim_shipping = dim_shipping[["Shipping_Mode_Key", "Shipping Mode", "Delivery Status"]]
# dim_shipping.to_csv(f"{output_dir}/dim_shipping.csv", index=False)
dim_shipping

,Shipping_Mode_Key,Shipping Mode,Delivery Status
0,e37562d8db525491802c26c748b9aa9c,Standard Class,Advance shipping
1,1206bf1f2853047e080eea5b1e2e3013,Standard Class,Late delivery
2,93809a49d1ee205e24449837b72d8caf,Standard Class,Shipping on time
3,3b67c90ab4e216c2cb9b13a46de07012,Standard Class,Shipping canceled
4,1b6377d639a22d5edb735bc742472945,First Class,Late delivery
5,00793535fcdf7c3b209bee65f7f2dbe3,Second Class,Late delivery
6,b03ad8d364620d96a8b6d6ee59cee76b,Second Class,Shipping canceled
7,1ee99b0c1e7b80707a6ed41557677c10,Same Day,Shipping on time
8,691932d019292c744d0177980c35c715,Same Day,Late delivery
9,986b6d730519ae53c22a792010a41ca8,Same Day,Shipping canceled


In [17]:
# CREATE DIM_ORDER_DETAILS

dim_order_details = (
    df[["Order Id", "Order Status", "Market"]].drop_duplicates(subset=["Order Id"]).reset_index(drop=True)
)
dim_order_details.to_csv(f"{output_dir}/dim_order_details.csv", index=False)

In [ ]:
# BUILD FACT_ORDER_PERFORMANCE
# Map the newly created surrogate keys back to the flat dataset
# Merge Location Key
df = df.merge(
    dim_location,
    on=["Order Region", "Order Country", "Order State", "Order City"],
    how="left",
)

# Merge Shipping Key
df = df.merge(dim_shipping, on=["Shipping Mode", "Delivery Status"], how="left")
df["order_date_dt"] = pd.to_datetime(df["order date (DateOrders)"])
df["shipping_date_dt"] = pd.to_datetime(df["shipping date (DateOrders)"])

df["Order_Date_Key"] = df["order_date_dt"].dt.strftime("%Y%m%d").astype(int)
df["Shipping_Date_Key"] = df["shipping_date_dt"].dt.strftime("%Y%m%d").astype(int)

# Extract final Fact table columns
fact_columns = [
    # Foreign Keys
    "Customer Id",
    "Product Card Id",
    "Location_Key",
    "Shipping_Mode_Key",
    "Order Id",
    "Order_Date_Key",
    "Shipping_Date_Key",
    # Operational Facts
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "shipping_variance",
    "late_delivery_flag",
    "Order Item Quantity",
    # Financial Facts
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Order Item Discount",
]

fact_order_performance = df[fact_columns].copy()
fact_order_performance.to_csv(
    f"{output_dir}/fact_order_performance.csv", index=False
)